<a href="https://colab.research.google.com/github/ravi-asati/rav-writes-abinitio-to-spark/blob/main/join_enrich_transactions.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# join — enrich transactions with customers (try it live)

Runnable companion to the note: **[join — enrich transactions with customers](https://ravi-writes.pages.dev/playground/join-enrich-transactions)**.

`join` brings two tables together on a shared key — the everyday **fact + dimension** lookup (Ab Initio's **Join** component). This notebook wires up the data and the setup; **you write the join** in the Spark solution cells below.

## The data & the requirement

Two curated tables (the *values* matter, so we download them rather than fake them):

- **`customers.csv`** — the dimension: one row per customer (`customer_id, name, region, signup_date`).
- **`transactions.csv`** — the fact: one row per transaction, carrying a `customer_id` foreign key (`transaction_id, customer_id, amount, txn_date`).

Two messy bits are planted on purpose:

- **An orphan transaction** — row `5006` has `customer_id = 999`, which is **not** in `customers`.
- **A customer with no activity** — `106 · Farah Noor` never appears in `transactions`.

**Requirement:** produce an **enriched transaction report** — every transaction with its buyer's **`name`** and **`region`** attached, keyed on `customer_id`. Decide what should happen to the orphan (`5006`): should it survive the join, and if so, what do its `name`/`region` show?

In [1]:
# Download both curated datasets into /content (Spark can't read an https:// URL directly)
import os
os.makedirs("/content/ravi-writes/data/input", exist_ok=True)

BASE = "https://raw.githubusercontent.com/ravi-asati/rav-writes-abinitio-to-spark/main/datasets"
!wget -q $BASE/customers.csv     -O /content/ravi-writes/data/input/customers.csv
!wget -q $BASE/transactions.csv  -O /content/ravi-writes/data/input/transactions.csv

# peek at the raw files
!echo '--- customers.csv ---'      && cat /content/ravi-writes/data/input/customers.csv
!echo '--- transactions.csv ---'   && cat /content/ravi-writes/data/input/transactions.csv

--- customers.csv ---
customer_id,name,region,signup_date
101,Asha Menon,South,2023-02-14
102,Ben Carter,North,2023-06-01
103,Chen Wei,East,2024-01-09
104,Diego Alvarez,West,2024-03-22
105,Esha Kapoor,South,2024-05-30
106,Farah Noor,North,2024-07-18
--- transactions.csv ---
transaction_id,customer_id,amount,txn_date
5001,101,120.50,2024-08-01
5002,103,45.00,2024-08-01
5003,101,80.00,2024-08-02
5004,102,220.75,2024-08-02
5005,104,15.20,2024-08-03
5006,999,60.00,2024-08-03
5007,103,12.99,2024-08-04
5008,105,300.00,2024-08-04
5009,102,55.40,2024-08-05
5010,104,410.00,2024-08-05


## The Ab Initio solution

_Your turn — sketch the Ab Initio graph: two **Input File** components feeding a **Join** on `customer_id`, out to an **Output File**. Note which port the orphan takes (an inner Join drops it to `unused`; a left-outer keeps it)._

## The Spark solution — step by step

Cells 1–4 are the setup (run them as-is). **Write your join in cell 5.**

In [2]:
# 1. Install PySpark (one-time, in Colab's runtime)
!pip install -q pyspark

In [3]:
# 2. Imports & SparkSession
from pyspark.sql import SparkSession
from pyspark.sql.functions import col

spark = SparkSession.builder.appName("join-enrich-transactions").getOrCreate()

In [4]:
# 3. Load both CSVs into DataFrames (header row + let Spark infer the types)
IN = "/content/ravi-writes/data/input"

customers = spark.read.format("csv").option("header", True).option("inferSchema", True).load(f"{IN}/customers.csv")
transactions = spark.read.format("csv").option("header", True).option("inferSchema", True).load(f"{IN}/transactions.csv")

customers.show()
transactions.show()

+-----------+-------------+------+-----------+
|customer_id|         name|region|signup_date|
+-----------+-------------+------+-----------+
|        101|   Asha Menon| South| 2023-02-14|
|        102|   Ben Carter| North| 2023-06-01|
|        103|     Chen Wei|  East| 2024-01-09|
|        104|Diego Alvarez|  West| 2024-03-22|
|        105|  Esha Kapoor| South| 2024-05-30|
|        106|   Farah Noor| North| 2024-07-18|
+-----------+-------------+------+-----------+

+--------------+-----------+------+----------+
|transaction_id|customer_id|amount|  txn_date|
+--------------+-----------+------+----------+
|          5001|        101| 120.5|2024-08-01|
|          5002|        103|  45.0|2024-08-01|
|          5003|        101|  80.0|2024-08-02|
|          5004|        102|220.75|2024-08-02|
|          5005|        104|  15.2|2024-08-03|
|          5006|        999|  60.0|2024-08-03|
|          5007|        103| 12.99|2024-08-04|
|          5008|        105| 300.0|2024-08-04|
|          5

In [5]:
# 4. Check the schemas Spark inferred (are customer_id / amount the types you expect?)
customers.printSchema()
transactions.printSchema()

root
 |-- customer_id: integer (nullable = true)
 |-- name: string (nullable = true)
 |-- region: string (nullable = true)
 |-- signup_date: date (nullable = true)

root
 |-- transaction_id: integer (nullable = true)
 |-- customer_id: integer (nullable = true)
 |-- amount: double (nullable = true)
 |-- txn_date: date (nullable = true)



In [27]:
# 5. YOUR JOIN HERE — attach each transaction's name + region, keyed on customer_id.
#    Hint: transactions.join(customers, on="customer_id", how="...")  then .select(...) the report columns.
#    Try how="inner" vs how="left" and watch what happens to the orphan (5006).

report = transactions.\
                        join(customers,
                             transactions.customer_id == customers.customer_id,
                            how="left") \
                                       .select(transactions.transaction_id,
                                               transactions.amount,
                                               transactions.txn_date,
                                               transactions.customer_id,
                                               customers.name.alias("customer_name"),
                                               customers.region.alias("customer_region"),
                                               customers.signup_date.alias("customer_signup_date")) # replace me
report.show()

+--------------+------+----------+-----------+-------------+---------------+--------------------+
|transaction_id|amount|  txn_date|customer_id|customer_name|customer_region|customer_signup_date|
+--------------+------+----------+-----------+-------------+---------------+--------------------+
|          5001| 120.5|2024-08-01|        101|   Asha Menon|          South|          2023-02-14|
|          5002|  45.0|2024-08-01|        103|     Chen Wei|           East|          2024-01-09|
|          5003|  80.0|2024-08-02|        101|   Asha Menon|          South|          2023-02-14|
|          5004|220.75|2024-08-02|        102|   Ben Carter|          North|          2023-06-01|
|          5005|  15.2|2024-08-03|        104|Diego Alvarez|           West|          2024-03-22|
|          5006|  60.0|2024-08-03|        999|         NULL|           NULL|                NULL|
|          5007| 12.99|2024-08-04|        103|     Chen Wei|           East|          2024-01-09|
|          5008| 300

## Your turn

1. **Inner vs. left on the orphan.** Run the join as `"inner"`, then `"left"`. Which one silently drops transaction `5006`, and which keeps it with `null` name/region?
2. **Find the orphans.** After a left join, isolate the rows whose `name` came back `null` — your referential-integrity report.
3. **The other direction.** Which join lets `106 · Farah Noor` (a customer with zero transactions) appear in the output?
4. **Spend per region.** Group the enriched report by `region` and total `amount`. Where does the orphan's `60.00` land?